# Adding augmented data to the dataset

In [ ]:
from datasets import load_dataset, ClassLabel, DatasetDict, Dataset
import pandas as pd
from huggingface_hub import login
import os
from transformers import AutoTokenizer
from tqdm import tqdm
from dotenv import load_dotenv

tqdm.pandas()

rseed = 42



In [ ]:
load_dotenv()
login(os.getenv("HF-TOKEN"))

In [ ]:
df = load_dataset("DrinkIcedT/mbti_unbalanced")
df_train = df["train"].to_pandas()


In [ ]:
df_train.info()

In [ ]:
print(df_train["I"].value_counts())
print(df_train["N"].value_counts())
print(df_train["F"].value_counts())
print(df_train["P"].value_counts())

In [ ]:
bt_sample = pd.read_csv("C:/Users/Tim/Projects/MA/data/csv/subsamples_augmentation/bt_augmented_agg_pub.csv", sep='\t', quoting=1)
bt_sample.drop("post", axis = 1, inplace = True)

bt_sample.rename(columns={"post_augmented": "post"}, inplace =True)


In [ ]:
df_train = pd.concat([df_train, bt_sample])

In [ ]:
print(df_train["I"].value_counts())
print(df_train["N"].value_counts())
print(df_train["F"].value_counts())
print(df_train["P"].value_counts())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

lengths = [len(tokenizer.encode(t)) for t in df_train['post']]
print(f"Max lokal: {max(lengths)}")

In [ ]:
# counting tokens
df_train["token_count"] = df_train["post"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)

print(df_train["token_count"].describe())

I decided to keep only observation with > 50 tokens, to drop very short sentences. Since these are the aggregated sentences, this is basically dropping unusable sentences.

In [ ]:
df_train = df_train[df_train["token_count"] > 50]
print(df_train["token_count"].describe())

In [ ]:
print(df_train["I"].value_counts())
print(df_train["N"].value_counts())
print(df_train["F"].value_counts())
print(df_train["P"].value_counts())

In [ ]:
def split_posts(df): 
    # splitting up the aggregated posts
    df = df.assign(post=df["post"].str.split("</s>")).explode("post")

    # delete whitespace and empty posts
    df["post"] = df["post"].str.strip()
    df = df[df["post"] != ""].reset_index(drop=True)

    return df

df_val = df["validation"].to_pandas()
df_test = df["test"].to_pandas()

df_val = split_posts(df_val)
df_test = split_posts(df_test)

In [ ]:
df_val.head(10)

In [ ]:
df_train = Dataset.from_pandas(df_train, preserve_index=False).shuffle()
df_val = Dataset.from_pandas(df_val, preserve_index=False).shuffle()
df_test = Dataset.from_pandas(df_test, preserve_index=False).shuffle()
# note: I did not set a seed here, which I should have done. 
# However, 
# 1. the original split was done with a random seed
# 2. the final dataset will be available after publication


mbti_labels = ["ENFJ", "ENFP", "ENTJ", "ENTP", "ESFJ", "ESFP", "ESTJ", "ESTP", 
               "INFJ", "INFP", "INTJ", "INTP", "ISFJ", "ISFP", "ISTJ", "ISTP"]


new_features = df_train.features.copy()
new_features["labels"] = ClassLabel(names=mbti_labels)

df_hf_train = df_train.cast(new_features)
df_hf_val = df_val.cast(new_features)
df_hf_test = df_test.cast(new_features)

df_hfdict = DatasetDict({
    "train": df_hf_train,
    "test": df_hf_test,
    "validation": df_hf_val
})


df_hfdict.save_to_disk("..\data\mbti_balanced_pub_new")
df_hfdict.push_to_hub("DrinkIcedT/mbti_balanced_pub_new")